# Extracting calendar features from dates

A date contains more information than one number. This notebook extracts year, quarter, month, ISO week, day of week, day of month, day of year, weekend, and calendar-boundary features using pandas.

The example uses a small reproducible order dataset so the notebook can run without a separate download.

## Why calendar features matter

Sales, visits, support requests, and energy usage often vary by season, month, weekday, or month-end. A model cannot discover these meanings from a raw string reliably, so we make them explicit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid',context='notebook')

rng=np.random.default_rng(42)
orders=pd.DataFrame({'order_time':pd.date_range('2024-01-01',periods=730,freq='12h')})
orders['orders']=rng.poisson(lam=20,size=len(orders))
orders['revenue']=orders['orders']*(40+rng.normal(0,5,len(orders)))
orders.head()

## 1. Parse the date safely

Always convert strings to a datetime dtype before using the dt accessor. If the source format is ambiguous, specify dayfirst, yearfirst, or an explicit format. Invalid values should be investigated rather than silently treated as valid dates.

In [ ]:
orders['order_time']=pd.to_datetime(orders['order_time'],errors='coerce',utc=True)
print(orders['order_time'].dtype)
print('Invalid dates:',orders['order_time'].isna().sum())
display(orders.head())

## 2. Extract calendar components

The dt accessor exposes calendar fields. ISO week is useful for weekly reporting; day_of_week uses Monday=0 and Sunday=6; boolean boundary fields are useful for month-end and period-start behaviour.

In [ ]:
dt=orders['order_time'].dt
orders['year']=dt.year
orders['quarter']=dt.quarter
orders['month']=dt.month
orders['week']=dt.isocalendar().week.astype('int64')
orders['day_of_week']=dt.dayofweek
orders['day_name']=dt.day_name()
orders['day_of_month']=dt.day
orders['day_of_year']=dt.dayofyear
orders['is_weekend']=dt.dayofweek.ge(5)
orders['is_month_start']=dt.is_month_start
orders['is_month_end']=dt.is_month_end
orders['is_quarter_end']=dt.is_quarter_end
display(orders.head())

## 3. Visualise calendar patterns

Grouping by month and weekday reveals whether the generated features expose a meaningful pattern. The bars below are descriptive; a model evaluation is still required before using a feature in production.

In [ ]:
monthly=orders.groupby('month',as_index=False)['orders'].mean()
weekday=orders.groupby(['day_of_week','day_name'],as_index=False)['orders'].mean().sort_values('day_of_week')
fig,axes=plt.subplots(1,2,figsize=(14,4))
sns.barplot(data=monthly,x='month',y='orders',color='#4C78A8',ax=axes[0]); axes[0].set_title('Average orders by month')
sns.barplot(data=weekday,x='day_name',y='orders',color='#F58518',ax=axes[1]); axes[1].set_title('Average orders by weekday'); axes[1].tick_params(axis='x',rotation=35)
plt.tight_layout(); plt.show()

## 4. Feature checks and modelling notes

Numeric month and weekday values are convenient, but they can create an artificial distance: December=12 looks far from January=1. For repeating annual or weekly patterns, consider cyclical encoding in the time-features notebook. Treat names such as day_name as categorical and encode them in a model pipeline.

In [ ]:
calendar_columns=['year','quarter','month','week','day_of_week','day_of_month','day_of_year','is_weekend','is_month_start','is_month_end','is_quarter_end']
display(orders[calendar_columns].describe(include='all').T)
print('Feature columns created:',calendar_columns)

## Summary

We parsed a timestamp and extracted calendar fields that can represent trend, seasonality, weekday behaviour, and period boundaries. The transformation is deterministic, but the choice of which fields to keep should be validated against the prediction task and the information available at prediction time.